###  Why a Second Notebook for Final Data Preparation?

Due to the large size of the raw dataset (~58 million rows), the initial transformation and sampling were handled separately in `01_data_preparation.ipynb`.

This notebook focuses on:
- Performing final cleaning on the sampled dataset
- Creating time-aware features (lags, rolling stats, etc.)
- Structuring the dataset for downstream analytics and forecasting

This separation improves performance, ensures modularity, and allows rapid iteration during exploratory analysis and modeling.


In [1]:
import pandas as pd

## Dataset Loading for Spare Parts Inventory Analysis

To begin the demand analysis and forecasting pipeline, we load three key preprocessed datasets


In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

calendar = pd.read_csv(DATA_DIR / "Calendar_new.csv")
sales = pd.read_csv(DATA_DIR / "Sales_merged.csv")
sampled_sales = pd.read_csv(DATA_DIR / "Sampled_sales.csv")

print(calendar.shape)
print(sales.shape)
print(sampled_sales.shape)

(1969, 26)
(956500, 8)
(9565, 7)


In [3]:
import pandas as pd
from pathlib import Path

RAW_DATA_DIR = Path("../raw_data")

sell_prices = pd.read_csv(RAW_DATA_DIR / "sell_prices.csv")

print(sell_prices.shape)

(6841121, 4)


##  Data Cleaning: Index & Auto-Saved Column Removal

When exporting or manipulating large datasets in CSV format, pandas or spreadsheet tools (like Excel) often introduce unwanted columns such as:

- `Unnamed: 0`, `Unnamed: 1`, etc. — usually result from saving DataFrames with `index=True`.
- `index` — sometimes added manually or by earlier merges.

These columns can clutter the dataset and interfere with downstream analysis.

In this step, we:

- Removed all columns matching the regex pattern `^Unnamed`.
- Dropped any column explicitly named `index`.
- Reset the row indices of the DataFrames to ensure they are clean and continuous.

This ensures a clean slate for further transformation, modeling, or export.


In [4]:
print("Variables loaded:")
print("sales:", sales.shape)
print("calendar:", calendar.shape)

try:
    print("sell_prices:", sell_prices.shape)
except:
    print("sell_prices not loaded")

Variables loaded:
sales: (956500, 8)
calendar: (1969, 26)
sell_prices: (6841121, 4)


In [5]:
# Drop unnamed columns (usually index columns auto-saved)
sales = sales.loc[:, ~sales.columns.str.contains('^Unnamed')]
calendar = calendar.loc[:, ~calendar.columns.str.contains('^Unnamed')]
sell_prices = sell_prices.loc[:, ~sell_prices.columns.str.contains('^Unnamed')]

# Also drop any column literally named 'index'
sales= sales.drop(columns=['index'], errors='ignore')
calendar = calendar.drop(columns=['index'], errors='ignore')
sell_prices = sell_prices.drop(columns=['index'], errors='ignore')

# Optional: Reset index if previously saved with index
sales.reset_index(drop=True, inplace=True)
calendar.reset_index(drop=True, inplace=True)
sell_prices.reset_index(drop=True, inplace=True)


##  Merging Sales and Calendar Data

To enrich our sales data with temporal context, we performed a **left join** between the `sales` DataFrame and the `calendar` DataFrame using the common column `'d'` (which represents the day identifier in the M5 dataset).

### Key Steps:
- **Left Join (`how='left'`)**: Ensures all sales records are preserved, even if some dates are missing in the calendar.
- **Index Cleanup**:
  - Checked for and removed any accidental index columns such as `'Unnamed: 0'`, `'0'`, or matching the DataFrame's index name.
  - Ensured a clean row structure by resetting the index with `drop=True`.

### Outcome:
The resulting `merged_df` contains:
- All original sales information
- Merged temporal features (weekend, holiday, SNAP flags, etc.)
- Cleaned column structure for further processing


In [6]:
merged_df = pd.merge(sales, calendar, how='left', on='d')
# Check if the first column is just an index
if merged_df.columns[0] == merged_df.index.name or merged_df.columns[0].startswith('Unnamed') or merged_df.columns[0] == '0':
    merged_df = merged_df.iloc[:, 1:]  # Drop first column

# Reset index just to be safe
merged_df.reset_index(drop=True, inplace=True)
merged_df

,id,item_id,dept_id,cat_id,store_id,state_id,d,units_sold,date,wm_yr_wk,...,snap_flag,week_of_month,season,day_of_year,is_payday,is_working_Day,event_in_3days,event_in_7days,is_month_start,is_month_end
0,FOODS_3_180_CA_1_validation,FOODS_3_180,FOODS_3,FOODS,CA_1,CA,d_1,0,2011-01-29,11101,...,0,5,Winter,29,0,0,0,0,0,0
1,HOUSEHOLD_2_383_CA_3_validation,HOUSEHOLD_2_383,HOUSEHOLD_2,HOUSEHOLD,CA_3,CA,d_1,2,2011-01-29,11101,...,0,5,Winter,29,0,0,0,0,0,0
2,FOODS_3_409_CA_3_validation,FOODS_3_409,FOODS_3,FOODS,CA_3,CA,d_1,0,2011-01-29,11101,...,0,5,Winter,29,0,0,0,0,0,0
3,FOODS_1_097_CA_2_validation,FOODS_1_097,FOODS_1,FOODS,CA_2,CA,d_1,0,2011-01-29,11101,...,0,5,Winter,29,0,0,0,0,0,0
4,HOBBIES_1_272_TX_2_validation,HOBBIES_1_272,HOBBIES_1,HOBBIES,TX_2,TX,d_1,0,2011-01-29,11101,...,0,5,Winter,29,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
956495,FOODS_3_143_TX_1_validation,FOODS_3_143,FOODS_3,FOODS,TX_1,TX,d_1913,1,2016-04-24,11613,...,0,4,Spring,115,0,0,0,1,0,0
956496,HOBBIES_1_221_CA_1_validation,HOBBIES_1_221,HOBBIES_1,HOBBIES,CA_1,CA,d_1913,0,2016-04-24,11613,...,0,4,Spring,115,0,0,0,1,0,0
956497,HOUSEHOLD_2_508_WI_1_validation,HOUSEHOLD_2_508,HOUSEHOLD_2,HOUSEHOLD,WI_1,WI,d_1913,0,2016-04-24,11613,...,0,4,Spring,115,0,0,0,1,0,0
956498,FOODS_3_500_TX_1_validation,FOODS_3_500,FOODS_3,FOODS,TX_1,TX,d_1913,0,2016-04-24,11613,...,0,4,Spring,115,0,0,0,1,0,0


In [7]:
df_final = pd.merge(
    merged_df,
    sell_prices,
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='inner'
)

In [8]:
df_final.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'units_sold', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'is_event', 'is_weekend', 'snap_flag',
       'week_of_month', 'season', 'day_of_year', 'is_payday', 'is_working_Day',
       'event_in_3days', 'event_in_7days', 'is_month_start', 'is_month_end',
       'sell_price'],
      dtype='str')

### Revenue definition

In [9]:
df_final['revenue'] = df_final['units_sold'] * df_final['sell_price']


## Rename columns for Spare Parts Inventory Simulation

In [10]:
# Rename columns for Spare Parts Inventory Simulation
df_final.rename(columns={
    'item_id': 'part_id',
    'store_id': 'location_id',
    'sell_price': 'unit_cost',
    'units_sold': 'daily_demand_units',
    'dept_id': 'part_type',
    'cat_id': 'category',
    'state_id': 'region',
    'wm_yr_wk': 'year_week',
    'weekday': 'day_name',
    'wday': 'day_num'
}, inplace=True)


In [11]:
for col in df_final:
    print(df_final[col].value_counts())

id
HOUSEHOLD_2_383_CA_3_validation    1913
FOODS_2_368_TX_2_validation        1913
HOUSEHOLD_1_395_TX_3_validation    1913
HOUSEHOLD_1_537_CA_1_validation    1913
HOUSEHOLD_2_410_TX_2_validation    1913
                                   ... 
FOODS_2_096_CA_2_validation         275
HOUSEHOLD_1_441_CA_1_validation     240
FOODS_2_114_CA_2_validation         212
FOODS_2_248_TX_3_validation         205
FOODS_3_192_WI_2_validation         191
Name: count, Length: 500, dtype: int64
part_id
FOODS_1_005        5739
HOBBIES_1_100      3826
HOUSEHOLD_2_062    3826
HOBBIES_1_178      3826
HOUSEHOLD_1_021    3826
                   ... 
FOODS_2_096         275
HOUSEHOLD_1_441     240
FOODS_2_114         212
FOODS_2_248         205
FOODS_3_192         191
Name: count, Length: 462, dtype: int64
part_type
FOODS_3        213106
HOUSEHOLD_2    126000
HOUSEHOLD_1    103417
FOODS_2        100305
HOBBIES_1       90576
FOODS_1         52725
HOBBIES_2       49346
Name: count, dtype: int64
category
FOODS   

##  Category to Part Class Mapping

To support segmentation of products based on their criticality and movement patterns, we mapped high-level product categories to custom-defined **part classes** using a simple dictionary.

### Mapping Logic:
We used the `category_map` dictionary to classify each product:

- `"FOODS"` → `"Fast_Moving"`  
- `"HOUSEHOLD"` → `"Regular"`  
- `"HOBBIES"` → `"Optional"`  

This classification was applied to the `category` column of the `df_final` DataFrame:



In [12]:
category_map = {
    "FOODS": "Fast_Moving",
    "HOUSEHOLD": "Regular",
    "HOBBIES": "Optional"
}
df_final["part_class"] = df_final["category"].map(category_map)


In [13]:
df_final.isnull().sum().sort_values(ascending=False)

event_type_2          733970
event_name_2          733970
event_type_1          676577
event_name_1          676577
category                   0
part_type                  0
part_id                    0
id                         0
location_id                0
region                     0
d                          0
daily_demand_units         0
day_num                    0
day_name                   0
year_week                  0
date                       0
month                      0
year                       0
snap_CA                    0
snap_TX                    0
snap_WI                    0
is_event                   0
is_weekend                 0
snap_flag                  0
week_of_month              0
season                     0
day_of_year                0
is_payday                  0
is_working_Day             0
event_in_3days             0
event_in_7days             0
is_month_start             0
is_month_end               0
unit_cost                  0
revenue       

## Dropping unnecessary columns

In [14]:
df_final.drop(["event_name_2", "event_type_2"], axis=1, inplace=True)


In [15]:
df_final["event_name_1"] = df_final["event_name_1"].fillna("No Event")
df_final["event_type_1"] = df_final["event_type_1"].fillna("No Event")

In [16]:
df_final.isnull().sum().sort_values(ascending=False)

id                    0
part_id               0
part_type             0
category              0
location_id           0
region                0
d                     0
daily_demand_units    0
date                  0
year_week             0
day_name              0
day_num               0
month                 0
year                  0
event_name_1          0
event_type_1          0
snap_CA               0
snap_TX               0
snap_WI               0
is_event              0
is_weekend            0
snap_flag             0
week_of_month         0
season                0
day_of_year           0
is_payday             0
is_working_Day        0
event_in_3days        0
event_in_7days        0
is_month_start        0
is_month_end          0
unit_cost             0
revenue               0
part_class            0
dtype: int64

In [17]:
pd.set_option('display.max_rows', None)

print(df_final['d'].value_counts().sort_index())

d
d_1       155
d_10      173
d_100     233
d_1000    409
d_1001    409
d_1002    409
d_1003    409
d_1004    409
d_1005    409
d_1006    409
d_1007    409
d_1008    409
d_1009    409
d_101     233
d_1010    409
d_1011    409
d_1012    409
d_1013    409
d_1014    409
d_1015    409
d_1016    409
d_1017    409
d_1018    409
d_1019    409
d_102     233
d_1020    409
d_1021    409
d_1022    409
d_1023    409
d_1024    409
d_1025    409
d_1026    409
d_1027    409
d_1028    409
d_1029    409
d_103     233
d_1030    409
d_1031    409
d_1032    409
d_1033    409
d_1034    409
d_1035    409
d_1036    409
d_1037    409
d_1038    409
d_1039    409
d_104     233
d_1040    409
d_1041    409
d_1042    409
d_1043    409
d_1044    412
d_1045    412
d_1046    412
d_1047    412
d_1048    412
d_1049    412
d_105     233
d_1050    412
d_1051    414
d_1052    414
d_1053    414
d_1054    414
d_1055    414
d_1056    414
d_1057    414
d_1058    415
d_1059    415
d_106     233
d_1060    415
d_1061    415
d_10

In [18]:
print("\nPipeline Completed Successfully")


Pipeline Completed Successfully


# Detailed Summary: 02_data_preparation.ipynb

## 1.  Notebook Purpose and Context

This notebook continues the **data preparation pipeline** for the **Spare Parts Inventory Optimization** project. It loads preprocessed calendar, sales, and price datasets, removes superfluous columns, and merges time-based features to produce a **clean, analysis-ready DataFrame** for downstream forecasting and optimization tasks.

---

## 2.  Dataset Loading

The following three CSV files are loaded using `pandas.read_csv()`:

| Dataset        | File Name            | Description                                                  |
|----------------|----------------------|--------------------------------------------------------------|
| Calendar       | `Calendar_new.csv`   | Engineered features (e.g., weekend, season, events)         |
| Sales          | `Sampled_sales.csv`  | 1% stratified sample of long-format M5 sales data           |
| Price History  | `sell_prices.csv`    | Product prices by store and week                            |

These are stored in DataFrames: `calendar`, `sales`, and `sell_prices`.

---

## 3.  Data Cleaning – Removing Unwanted Index Columns

CSV exports often introduce unnecessary columns (e.g., `Unnamed: 0`, `index`). To clean the structure:

- **Dropped all columns matching**: `^Unnamed` using regex
- **Explicitly dropped**: Column named `index` if present
- **Reset indices**: `reset_index(drop=True)` used on all three DataFrames

This ensures a clean, zero-based, and reproducible tabular structure.

---

## 4.  Merging Sales with Calendar Features

The `sales` DataFrame is **enriched with calendar features** via a left-join:

- **Merge Key**: `d` (day identifier)
- **Join Type**: `how='left'` — ensures all sales rows are retained
- **Post-Merge Cleaning**:
  - Dropped any residual `Unnamed` or `index` columns
  - Reset index of the merged DataFrame for continuity

---

## 5.  Structure of the Final Merged DataFrame

After merging, the final `merged_df` has:

- **Shape**: `583,465 rows × 33 columns`
- **Includes**:

###  Identifiers
- `id`, `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id`

###  Sales Metrics
- `d` (M5-style day code)
- `units_sold`

###  Date Attributes
- `date`, `wm_yr_wk`

###  Engineered Calendar Features
| Feature Category          | Example Columns                                   |
|---------------------------|---------------------------------------------------|
| Event Flags               | `is_event`, `event_type_1`, `event_in_3days`     |
| Seasonality & Holidays    | `season`, `is_weekend`, `is_payday`              |
| Promotion Indicators      | `snap_CA`, `snap_TX`, `snap_WI`                  |
| Temporal Positioning      | `day_of_year`, `week_of_month`, `wm_yr_wk`       |
| Month Boundary Flags      | `is_month_start`, `is_month_end`                 |
| Workday Indicators        | `is_working_Day`                                  |

---

## 6.  Outcome and Next Steps

###  Clean, Unified Dataset
All temporal, event, and promotional features are now aligned with historical sales.

###  Reproducibility
Cleaning and merging steps follow a structured, repeatable format.

###  Ready for Modeling
This merged dataset is now fully prepared for:
- Feature selection
- Model training (forecasting)
- Service level simulation and inventory optimization

---

##  Summary

This notebook finalizes the **data preparation pipeline** by producing a **highly structured, clean, and feature-rich dataset** for spare parts demand forecasting. It bridges the gap between raw data and modeling, enabling robust time-series analytics in upcoming notebooks.


# Final Cleaned Processed Dataset for Actual EDA

In [19]:
df_final.to_csv("SparePartsInventory.csv", index=False)
